In [13]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
import numpy as np




In [14]:

# 1. Load both datasets
train_df = pd.read_csv("../train.csv")
test_df = pd.read_csv("../test.csv")

# 2. Separate training features, target, and test IDs
X_train = train_df.drop(
    columns=["Price_INR_Lakhs", "Property_ID"]
)
y_train = train_df["Price_INR_Lakhs"]

property_ids = test_df["Property_ID"].copy()

# Keep only the same feature columns used during training,
# in the same order
X_test = test_df.drop(
    columns=["Property_ID", "Price_INR_Lakhs"],
    errors="ignore"
).reindex(columns=X_train.columns)

# 3. Define feature groups
categorical_features = [
    "City",
    "Locality_Type",
    "Property_Type",
    "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

# 4. Build preprocessing pipeline
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
], sparse_threshold=0)

# 5. Create the selected model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        learning_rate=0.1,
        max_iter=100,
        max_leaf_nodes=15,
        random_state=42
    ))
])

# 6. Train on all 80,000 labelled rows
model.fit(X_train, y_train)

# 7. Predict prices for the 20,000 unlabelled test rows
predictions = model.predict(X_test)

# Property prices cannot be negative
predictions = np.maximum(predictions, 0)
# 8. Save predictions with their property IDs
output_df = pd.DataFrame({
    "Property_ID": property_ids,
    "Predicted_Price_INR_Lakhs": predictions
})

output_df.to_csv("../test_predictions.csv", index=False)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nFirst 10 predictions:")
print(output_df.head(10).round(2).to_string(index=False))

print("\nPredictions saved to ../test_predictions.csv")


Training rows: 80000
Test rows: 20000

First 10 predictions:
Property_ID  Predicted_Price_INR_Lakhs
PROP-009209                     182.37
PROP-053772                     116.70
PROP-001197                     394.08
PROP-043368                      91.47
PROP-058815                     143.72
PROP-038284                     224.33
PROP-007780                     213.94
PROP-050321                     146.80
PROP-050246                      74.06
PROP-025067                     104.30

Predictions saved to ../test_predictions.csv


In [15]:
print("\nPrediction validation")
print("Rows:", len(output_df))
print("Duplicate IDs:", output_df["Property_ID"].duplicated().sum())
print("Missing predictions:",
      output_df["Predicted_Price_INR_Lakhs"].isna().sum())
print("Minimum prediction:",
      output_df["Predicted_Price_INR_Lakhs"].min())
print("Maximum prediction:",
      output_df["Predicted_Price_INR_Lakhs"].max())
print("All predictions finite:",
      output_df["Predicted_Price_INR_Lakhs"].map(
          lambda x: pd.notna(x) and abs(x) != float("inf")
      ).all())


Prediction validation
Rows: 20000
Duplicate IDs: 0
Missing predictions: 0
Minimum prediction: 0.0
Maximum prediction: 1239.1358321095981
All predictions finite: True
